# Math 189 Final Project
## Can Neighborhood Wealth Predict County-Level Standardized Test Scores in California?

**Data Sources:** CAASPP (2024-25), CDE FRPM (2024-25), IPUMS ACS 5-Year (2023)

---

### Table of Contents
1. Imports & Setup
2. Data Acquisition — CAASPP Test Scores
3. Data Acquisition — FRPM Poverty Data
4. Data Acquisition — IPUMS ACS Census Data
5. Data Merging & Cleaning
6. Exploratory Data Analysis (EDA)
7. California County Heatmap
8. Two-Sample KS Test
9. VIF Analysis
10. Variable Selection
11. Variance Decomposition
12. Linear Regression
13. Regression Diagnostics

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import gzip
import os
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.preprocessing import StandardScaler
from matplotlib.patches import Patch

# Plot settings
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'


In [2]:
# Load raw CAASPP file from zip archive
with zipfile.ZipFile('sb_ca2025_1_csv_v1.zip', 'r') as z:
    with z.open('sb_ca2025_1_csv_v1.txt') as f:
        df_raw = pd.read_csv(f, delimiter='^', dtype=str, low_memory=False, encoding='latin-1')

print(f"Raw CAASPP data: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")

Raw CAASPP data: 102,044 rows, 69 columns


In [3]:
# Filter to county-level ELA and Math
county = df_raw[
    (df_raw['Type ID'] == '5') &
    (df_raw['Grade'] == '13')
].copy()

county = county[['County Code', 'Test ID', 'Total Students Tested', 'Percentage Standard Met and Above']]
county['Percentage Standard Met and Above'] = pd.to_numeric(county['Percentage Standard Met and Above'], errors='coerce')
county['Total Students Tested'] = pd.to_numeric(county['Total Students Tested'], errors='coerce')

county_wide = county.pivot_table(
    index='County Code',
    columns='Test ID',
    values='Percentage Standard Met and Above',
    aggfunc='first'
).reset_index()

county_wide.columns.name = None
county_wide = county_wide.rename(columns={'1': 'pct_met_ELA', '2': 'pct_met_Math'})

In [4]:
# Add county names
with zipfile.ZipFile('sb_ca2025_1_csv_v1.zip', 'r') as z:
    with z.open('sb_ca2025entities_csv.txt') as f:
        entities = pd.read_csv(f, delimiter='^', dtype=str, low_memory=False, encoding='latin-1')

county_names = entities[entities['Type ID'] == '5'][['County Code', 'County Name']].drop_duplicates()
county_wide = county_wide.merge(county_names, on='County Code', how='left')
county_wide = county_wide[['County Code', 'County Name', 'pct_met_ELA', 'pct_met_Math']]

In [5]:
print(county_wide.to_string(index=False))

County Code     County Name  pct_met_ELA  pct_met_Math
         01         Alameda        56.16         48.15
         02          Alpine        43.24         29.41
         03          Amador        35.79         23.49
         04           Butte        46.38         33.23
         05       Calaveras        34.21         23.18
         06          Colusa        35.82         23.29
         07    Contra Costa        51.98         42.02
         08       Del Norte        34.02         21.13
         09       El Dorado        57.75         46.07
         10          Fresno        46.44         34.09
         11           Glenn        31.92         21.68
         12        Humboldt        42.54         31.92
         13        Imperial        41.59         27.11
         14            Inyo        35.75         28.70
         15            Kern        39.94         24.26
         16           Kings        46.38         31.99
         17            Lake        26.07         15.87
         1

In [6]:
# Check sheet names in FRPM Excel file
xl = pd.ExcelFile('frpm2425.xlsx')
print(xl.sheet_names)

['Title Page', 'FRPM School-Level Data', 'Data Field Descriptions']


In [7]:
# FRPM 2024-25 — Load Raw File
frpm_raw = pd.read_excel('frpm2425.xlsx', sheet_name='FRPM School-Level Data', dtype=str, header=1)

In [8]:
# Select and Rename Relevant Columns
frpm = frpm_raw[['County Code', 'County Name', 'Enrollment \n(K-12)', 'Free Meal \nCount \n(K-12)']].copy()
frpm.columns = ['County Code', 'County Name', 'enrollment', 'free_meal_count']
frpm['enrollment'] = pd.to_numeric(frpm['enrollment'], errors='coerce')
frpm['free_meal_count'] = pd.to_numeric(frpm['free_meal_count'], errors='coerce')

In [9]:
# Aggregate to County Level
frpm_county = frpm.groupby(['County Code', 'County Name']).sum().reset_index()
frpm_county['frpm_pct_free'] = (frpm_county['free_meal_count'] / frpm_county['enrollment'] * 100).round(2)

In [10]:
# Merge with CAASPP dataset
merged = county_wide.merge(
    frpm_county[['County Code', 'frpm_pct_free']],
    on='County Code',
    how='left'
)

In [11]:
# Read in Census data
with gzip.open('usa_00003.csv.gz', 'rt') as f:
    acs_raw2 = pd.read_csv(f, dtype=str, low_memory=False)

print(acs_raw2.shape)
print(acs_raw2.columns.tolist())

(15912393, 22)
['YEAR', 'MULTYEAR', 'SAMPLE', 'SERIAL', 'CBSERIAL', 'HHWT', 'CLUSTER', 'STATEFIP', 'COUNTYFIP', 'STRATA', 'GQ', 'OWNERSHP', 'OWNERSHPD', 'HHINCOME', 'FOODSTMP', 'VALUEH', 'PERNUM', 'PERWT', 'EDUC', 'EDUCD', 'INCWELFR', 'POVERTY']


In [12]:
# Filter to California only and drop unidentifiable counties
acs2 = acs_raw2[
    (acs_raw2['STATEFIP'] == '6') &
    (acs_raw2['COUNTYFIP'] != '0')
].copy()

# Convert numeric columns
acs2['HHINCOME']  = pd.to_numeric(acs2['HHINCOME'], errors='coerce').replace(9999999, np.nan)
acs2['POVERTY']   = pd.to_numeric(acs2['POVERTY'], errors='coerce')
acs2['EDUCD']     = pd.to_numeric(acs2['EDUCD'], errors='coerce')
acs2['HHWT']      = pd.to_numeric(acs2['HHWT'], errors='coerce')
acs2['PERWT']     = pd.to_numeric(acs2['PERWT'], errors='coerce')
acs2['VALUEH']    = pd.to_numeric(acs2['VALUEH'], errors='coerce').replace(9999999, np.nan)
acs2['INCWELFR']  = pd.to_numeric(acs2['INCWELFR'], errors='coerce').replace(99999, np.nan)
acs2['FOODSTMP']  = pd.to_numeric(acs2['FOODSTMP'], errors='coerce')
acs2['OWNERSHP']  = pd.to_numeric(acs2['OWNERSHP'], errors='coerce')

print(acs2.shape)
print(acs2['COUNTYFIP'].nunique(), 'counties')

(1783424, 22)
34 counties


In [13]:
# Median household income (weighted)
income = acs2[acs2['HHINCOME'].notna()].groupby('COUNTYFIP').apply(
    lambda x: np.average(x['HHINCOME'], weights=x['HHWT']), include_groups=False
).reset_index()
income.columns = ['COUNTYFIP', 'median_income']

# Poverty rate — below 100 = below poverty line
poverty = acs2.groupby('COUNTYFIP').apply(
    lambda x: np.average(x['POVERTY'] < 100, weights=x['PERWT']), include_groups=False
).reset_index()
poverty.columns = ['COUNTYFIP', 'poverty_rate']

# % adults with bachelor's or higher
educ = acs2[acs2['EDUCD'].notna()].groupby('COUNTYFIP').apply(
    lambda x: np.average(x['EDUCD'] >= 101, weights=x['PERWT']), include_groups=False
).reset_index()
educ.columns = ['COUNTYFIP', 'pct_bachelors_plus']

# % households on SNAP/food stamps (FOODSTMP == 2 means yes)
foodstmp = acs2.groupby('COUNTYFIP').apply(
    lambda x: np.average(x['FOODSTMP'] == 2, weights=x['HHWT']), include_groups=False
).reset_index()
foodstmp.columns = ['COUNTYFIP', 'pct_foodstmp']

# Homeownership rate (OWNERSHP == 1 or 2 means owned)
ownershp = acs2[acs2['OWNERSHP'].notna()].groupby('COUNTYFIP').apply(
    lambda x: np.average(x['OWNERSHP'].isin([1, 2]), weights=x['HHWT']), include_groups=False
).reset_index()
ownershp.columns = ['COUNTYFIP', 'homeownership_rate']

# Median house value (weighted)
valueh = acs2[acs2['VALUEH'].notna()].groupby('COUNTYFIP').apply(
    lambda x: np.average(x['VALUEH'], weights=x['HHWT']), include_groups=False
).reset_index()
valueh.columns = ['COUNTYFIP', 'median_house_value']

# Welfare income rate — % of people receiving any welfare income
incwelfr = acs2[acs2['INCWELFR'].notna()].groupby('COUNTYFIP').apply(
    lambda x: np.average(x['INCWELFR'] > 0, weights=x['PERWT']), include_groups=False
).reset_index()
incwelfr.columns = ['COUNTYFIP', 'pct_welfare']

# Merge all together
acs_county2 = income.merge(poverty, on='COUNTYFIP')\
                    .merge(educ, on='COUNTYFIP')\
                    .merge(foodstmp, on='COUNTYFIP')\
                    .merge(ownershp, on='COUNTYFIP')\
                    .merge(valueh, on='COUNTYFIP')\
                    .merge(incwelfr, on='COUNTYFIP')

print(acs_county2.shape)
print(acs_county2.head())

(34, 8)
  COUNTYFIP  median_income  poverty_rate  pct_bachelors_plus  pct_foodstmp  \
0         1  193855.427932      0.103755            0.384415      0.104060   
1       107   93570.846285      0.175032            0.097486      0.296714   
2       111  152829.611773      0.089128            0.256261      0.107451   
3       113  133577.927130      0.182771            0.288341      0.142814   
4        13  189351.205012      0.083536            0.327224      0.100044   

   homeownership_rate  median_house_value  pct_welfare  
0            0.977600        1.130251e+06     0.021909  
1            0.987182        3.852422e+05     0.026690  
2            0.982702        8.260923e+05     0.013045  
3            0.957775        6.787214e+05     0.015338  
4            0.989754        9.903505e+05     0.014255  


In [14]:
# Verify OWNERSHP codes: 1 = owned, 2 = rented, 0 = N/A (group quarters)
print(acs2['OWNERSHP'].value_counts())

OWNERSHP
1    1074544
2     623029
0      85851
Name: count, dtype: int64


In [15]:
# Fix homeownership rate — initial calculation was incorrect
# OWNERSHP == 1 means owned, OWNERSHP == 2 means rented, 0 = group quarters (exclude)
ownershp = acs2[acs2['OWNERSHP'] != 0].groupby('COUNTYFIP').apply(
    lambda x: np.average(x['OWNERSHP'] == 1, weights=x['HHWT']), include_groups=False
).reset_index()
ownershp.columns = ['COUNTYFIP', 'homeownership_rate']

# Rebuild acs_county2 with corrected homeownership rate
acs_county2 = income.merge(poverty, on='COUNTYFIP')\
                    .merge(educ, on='COUNTYFIP')\
                    .merge(foodstmp, on='COUNTYFIP')\
                    .merge(ownershp, on='COUNTYFIP')\
                    .merge(valueh, on='COUNTYFIP')\
                    .merge(incwelfr, on='COUNTYFIP')

print(ownershp.head(10))

  COUNTYFIP  homeownership_rate
0         1            0.570250
1       107            0.581021
2       111            0.636460
3       113            0.539751
4        13            0.679501
5        17            0.772462
6        19            0.561870
7        23            0.589934
8        25            0.579215
9        29            0.596346


In [16]:
# Check columns in merged CAASPP + FRPM dataset before joining Census data
print(merged.columns.tolist())

['County Code', 'County Name', 'pct_met_ELA', 'pct_met_Math', 'frpm_pct_free']


In [17]:
# CDE uses its own 2-digit county codes while IPUMS uses 3-digit federal FIPS codes
# A manual crosswalk is required, a naive join on county code produces incorrect matches
crosswalk = {
    '01': '001', '02': '003', '03': '005', '04': '007', '05': '009',
    '06': '011', '07': '013', '08': '015', '09': '017', '10': '019',
    '11': '021', '12': '023', '13': '025', '14': '027', '15': '029',
    '16': '031', '17': '033', '18': '035', '19': '037', '20': '039',
    '21': '041', '22': '043', '23': '045', '24': '047', '25': '049',
    '26': '051', '27': '053', '28': '055', '29': '057', '30': '059',
    '31': '061', '32': '063', '33': '065', '34': '067', '35': '069',
    '36': '071', '37': '073', '38': '075', '39': '077', '40': '079',
    '41': '081', '42': '083', '43': '085', '44': '087', '45': '089',
    '46': '091', '47': '093', '48': '095', '49': '097', '50': '099',
    '51': '101', '52': '103', '53': '105', '54': '107', '55': '109',
    '56': '111', '57': '113', '58': '115'
}

merged['COUNTYFIP_3'] = merged['County Code'].map(crosswalk)
acs_county2['COUNTYFIP_3'] = acs_county2['COUNTYFIP'].astype(str).str.zfill(3)

final2 = merged.merge(
    acs_county2[['COUNTYFIP_3', 'median_income', 'poverty_rate', 'pct_bachelors_plus',
                 'pct_foodstmp', 'homeownership_rate', 'median_house_value', 'pct_welfare']],
    on='COUNTYFIP_3',
    how='inner'
)
print(final2.shape)
print(final2.head())

(34, 13)
  County Code   County Name  pct_met_ELA  pct_met_Math  frpm_pct_free  \
0          01       Alameda        56.16         48.15          40.91   
1          04         Butte        46.38         33.23          54.39   
2          07  Contra Costa        51.98         42.02          38.35   
3          09     El Dorado        57.75         46.07          28.91   
4          10        Fresno        46.44         34.09          60.99   

  COUNTYFIP_3  median_income  poverty_rate  pct_bachelors_plus  pct_foodstmp  \
0         001  193855.427932      0.103755            0.384415      0.104060   
1         007  109947.537301      0.179910            0.223573      0.177883   
2         013  189351.205012      0.083536            0.327224      0.100044   
3         017  165400.949140      0.086965            0.292535      0.067082   
4         019  105361.665528      0.188770            0.156259      0.253679   

   homeownership_rate  median_house_value  pct_welfare  
0            0

In [18]:
# Final dataset sanity check
print(f"Missing values:\n{final2.isnull().sum()}")
print(f"\nDescriptive stats:")
print(final2[['pct_met_ELA', 'pct_met_Math', 'frpm_pct_free', 'median_income', 
              'poverty_rate', 'pct_bachelors_plus', 'pct_foodstmp', 
              'homeownership_rate', 'median_house_value', 'pct_welfare']].describe().round(3))

Missing values:
County Code           0
County Name           0
pct_met_ELA           0
pct_met_Math          0
frpm_pct_free         0
COUNTYFIP_3           0
median_income         0
poverty_rate          0
pct_bachelors_plus    0
pct_foodstmp          0
homeownership_rate    0
median_house_value    0
pct_welfare           0
dtype: int64

Descriptive stats:
       pct_met_ELA  pct_met_Math  frpm_pct_free  median_income  poverty_rate  \
count       34.000        34.000         34.000         34.000        34.000   
mean        47.884        36.331         51.577     145894.157         0.134   
std          7.266         8.994         12.782      46534.042         0.044   
min         36.530        23.150         28.910      87272.828         0.066   
25%         42.315        29.803         43.255     110375.167         0.098   
50%         46.380        33.725         50.205     136345.883         0.124   
75%         53.370        41.980         61.762     167514.703         0.175   

In [19]:
# Save final merged dataset to CSV for use in subsequent analyses
final2.to_csv('ca_county_schools_final.csv', index=False)
print("Saved")

Saved


---